# Colab AI Chat Connector — Gemini 3.8 Flash\n\nTap ▶ on the next cell. The app provides a short-lived pairing token.\n\nThis connector **forces `google/gemini-3.8-flash`**. For prompts asking about today's prices, latest news, the internet, or current information, it gathers fresh web/market evidence first and then gives that evidence to Gemini 3.8 Flash.\n\nThe current `google.colab.ai.generate_text()` wrapper does not expose Gemini's native Google Search grounding tool directly, so this notebook supplies its own live research layer.\n

In [ ]:
import json, time, urllib.request, urllib.error, urllib.parse, re, html
from datetime import datetime, timezone

try:
    from google.colab import ai, output
except Exception as exc:
    raise RuntimeError("This connector must run inside Google Colab.") from exc

SUPABASE_URL = "https://udyqxlnexnxhmslsebre.supabase.co"
SUPABASE_KEY = "sb_publishable_IgyY0SZrN1tZUPNRnHWVrw_OOZwEsrH"
MODEL_NAME = "google/gemini-3.8-flash"

def http_get(url, timeout=20):
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_0 like Mac OS X) AppleWebKit/605.1.15 Safari/604.1",
            "Accept-Language": "en-US,en;q=0.9",
        },
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")

def strip_html(value):
    value = re.sub(r"<script[\\s\\S]*?</script>", " ", value, flags=re.I)
    value = re.sub(r"<style[\\s\\S]*?</style>", " ", value, flags=re.I)
    value = re.sub(r"<[^>]+>", " ", value)
    value = html.unescape(value)
    return re.sub(r"\\s+", " ", value).strip()

def web_search(query, limit=5):
    try:
        url = "https://html.duckduckgo.com/html/?" + urllib.parse.urlencode({"q": query})
        page = http_get(url)
        items = []
        blocks = re.findall(r'<div class="result[^"]*"[^>]*>([\\s\\S]*?)</div>\\s*</div>', page, flags=re.I)
        for block in blocks:
            m = re.search(r'class="result__a"[^>]*href="([^"]+)"[^>]*>([\\s\\S]*?)</a>', block, flags=re.I)
            if not m:
                continue
            href = html.unescape(m.group(1))
            if "uddg=" in href:
                try:
                    href = urllib.parse.parse_qs(urllib.parse.urlparse(href).query).get("uddg", [href])[0]
                except Exception:
                    pass
            sm = re.search(r'class="result__snippet"[^>]*>([\\s\\S]*?)</', block, flags=re.I)
            items.append({
                "title": strip_html(m.group(2)),
                "url": href,
                "snippet": strip_html(sm.group(1)) if sm else "",
            })
            if len(items) >= limit:
                break
        return items
    except Exception as exc:
        return [{"title": "Search error", "url": "", "snippet": str(exc)}]

def yahoo_quote(symbol):
    try:
        q = urllib.parse.quote(symbol, safe="")
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{q}?range=1d&interval=1m&includePrePost=false"
        data = json.loads(http_get(url))
        result = data["chart"]["result"][0]
        meta = result.get("meta", {})
        closes = result.get("indicators", {}).get("quote", [{}])[0].get("close", [])
        last_close = next((x for x in reversed(closes) if x is not None), None)
        price = meta.get("regularMarketPrice")
        if price is None:
            price = last_close
        prev = meta.get("chartPreviousClose") or meta.get("previousClose")
        change = (price - prev) if price is not None and prev is not None else None
        pct = (change / prev * 100) if change is not None and prev else None
        return {
            "symbol": symbol,
            "price": price,
            "currency": meta.get("currency"),
            "exchange": meta.get("exchangeName"),
            "previous_close": prev,
            "change": change,
            "percent_change": pct,
            "market_state": meta.get("marketState"),
            "regular_market_time": meta.get("regularMarketTime"),
            "source": f"https://finance.yahoo.com/quote/{symbol}",
        }
    except Exception as exc:
        return {"symbol": symbol, "error": str(exc)}

def finance_context(prompt):
    p = prompt.lower()
    symbols = []
    if any(k in p for k in ["glm", "智谱", "z.ai", "zai"]):
        symbols = ["2513.HK", "02513.HK"]
    results = []
    for symbol in symbols:
        q = yahoo_quote(symbol)
        if q.get("price") is not None:
            results.append(q)
            break
    return results

def needs_live_research(prompt):
    p = prompt.lower()
    hints = [
        "today", "current", "latest", "now", "live", "price", "stock", "market",
        "news", "search", "research", "internet", "web", "今天", "现在", "最新",
        "股价", "股票", "行情", "新闻", "搜索", "查一下"
    ]
    return any(h in p for h in hints)

def build_research_prompt(prompt):
    if not needs_live_research(prompt):
        return prompt
    finance = finance_context(prompt)
    search = web_search(prompt, limit=6)
    evidence = {
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "finance": finance,
        "web_results": search,
    }
    return f"""You are Gemini 3.8 Flash. Answer the user's request using the fresh evidence below.
The evidence was retrieved immediately before this answer.
Do not say you cannot access current information: current web/market evidence has already been provided to you.
For market prices, prefer the structured finance quote over search snippets.
State the currency and timestamp/context when available.
If evidence conflicts or is insufficient, say exactly what is uncertain.
Include source URLs from the evidence at the end under Sources.

FRESH EVIDENCE:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

USER REQUEST:
{prompt}
"""

def rpc(name, payload):
    body = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        SUPABASE_URL + "/rest/v1/rpc/" + name,
        data=body,
        headers={
            "apikey": SUPABASE_KEY,
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw = resp.read().decode("utf-8")
            return json.loads(raw) if raw else None
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Connector HTTP {exc.code}: {detail}") from None

def get_pair_token():
    try:
        value = output.eval_js("navigator.clipboard.readText()")
        if isinstance(value, str):
            value = value.strip()
            if value.startswith("AIWB-") and len(value) >= 29:
                print("Pairing token read from clipboard ✓")
                return value
    except Exception:
        pass
    print("Clipboard access was blocked by the browser.")
    return input("Paste the pairing token copied by the app: ").strip()

pair_token = get_pair_token()
started_at = datetime.now(timezone.utc).isoformat()

def heartbeat(status, include_started=False):
    payload = {"p_token": pair_token, "p_status": status}
    if include_started:
        payload["p_started_at"] = started_at
    ok = rpc("aiwb_pair_heartbeat", payload)
    if ok is not True:
        raise RuntimeError("Pairing token is invalid or expired. Return to the Vercel app and tap Enable Colab again.")

def claim_job():
    jobs = rpc("aiwb_pair_claim_job", {"p_token": pair_token})
    return jobs or []

def finish_job(job_id, result=None, error=None):
    return rpc("aiwb_pair_finish_job", {
        "p_token": pair_token,
        "p_job_id": job_id,
        "p_result": result,
        "p_error": error,
    })

heartbeat("starting", include_started=True)
print("Testing Google Colab AI...")
try:
    models = ai.list_models()
    if MODEL_NAME not in models:
        raise RuntimeError("Gemini 3.8 Flash is not currently enabled for this Colab account. Available models: " + ", ".join(models))
    test = ai.generate_text("Reply with exactly: COLAB_AI_OK", model_name=MODEL_NAME)
    test_text = getattr(test, "text", None) or str(test)
    if "COLAB_AI_OK" not in test_text:
        print("AI responded, but with unexpected text:", test_text[:120])
except Exception as exc:
    try:
        heartbeat("error")
    except Exception:
        pass
    raise RuntimeError("Colab AI test failed: " + str(exc)) from exc

heartbeat("ready")
print("\nCONNECTED ✓")
print("Return to https://colab-ai-workbench.vercel.app")
print("Model: Gemini 3.8 Flash")
print("Live research: enabled for current/news/market/search prompts")
print("No email or password is stored in Colab.\n")

last_job = time.monotonic()
try:
    while True:
        heartbeat("ready")
        jobs = claim_job()
        if jobs:
            job = jobs[0]
            try:
                heartbeat("busy")
                research_prompt = build_research_prompt(job["prompt"])
                response = ai.generate_text(research_prompt, model_name=MODEL_NAME)
                answer = getattr(response, "text", None) or str(response)
                finish_job(job["id"], result=answer[:100000])
            except Exception as exc:
                finish_job(job["id"], error=str(exc)[:2000])
            finally:
                last_job = time.monotonic()
                heartbeat("ready")
        else:
            if time.monotonic() - last_job > 1800:
                print("Idle 30 minutes — connector stopped. Tap Enable Colab again when needed.")
                break
            time.sleep(1)
finally:
    try:
        heartbeat("stopped")
    except Exception:
        pass
